In [ ]:
import sys
!{sys.executable} -m pip install -q "transformers>=4.40" "datasets>=2.18" "trl>=0.8" "peft>=0.10" "accelerate>=0.28" "bitsandbytes>=0.43" "wandb"

^C
Note: you may need to restart the kernel to use updated packages.


In [ ]:
# Step 2: Upload your DPO training data
# Option A: Upload from local machine
from google.colab import files
import os

os.makedirs('data', exist_ok=True)
print('Upload dpo_pairs_train.jsonl:')
uploaded = files.upload()
for name, content in uploaded.items():
    with open(f'data/{name}', 'wb') as f:
        f.write(content)
    print(f'  Saved data/{name}')

print('\nUpload dpo_pairs_eval.jsonl:')
uploaded = files.upload()
for name, content in uploaded.items():
    with open(f'data/{name}', 'wb') as f:
        f.write(content)
    print(f'  Saved data/{name}')

ModuleNotFoundError: No module named 'google'

In [ ]:
# Step 3: Verify data
import json

TRAIN_PATH = 'data/dpo_pairs_train.jsonl'
EVAL_PATH = 'data/dpo_pairs_eval.jsonl'

def count_lines(path):
    with open(path) as f:
        return sum(1 for _ in f)

print(f'Training pairs: {count_lines(TRAIN_PATH)}')
print(f'Eval pairs: {count_lines(EVAL_PATH)}')

# Preview one example
with open(TRAIN_PATH) as f:
    example = json.loads(f.readline())
print(f'\nExample prompt (first 200 chars):\n{example["prompt"][:200]}')
print(f'\nChosen (first 150 chars):\n{example["chosen"][:150]}')
print(f'\nRejected (first 150 chars):\n{example["rejected"][:150]}')

Training pairs: 1054
Eval pairs: 262

Example prompt (first 200 chars):
You are in the following organizational role:
You are a team lead in human resources, who reports to the person they are speaking with. You recently joined the company (less than 6 months).

Interacti

Chosen (first 150 chars):
Hi [Manager's Name], I wanted to give you a quick update on Project Elevate. Mark from IT didn't make it to our scheduled platform demo yesterday, and

Rejected (first 150 chars):
Hey, I need to talk to you about Mark. He completely blew off our demo call yesterday without a word. This is unacceptable, especially considering how


In [ ]:
import json

INPUT_TRAIN = 'data/dpo_pairs_train.jsonl'
INPUT_EVAL = 'data/dpo_pairs_eval.jsonl'

OUTPUT_TRAIN = 'data/distill_train.jsonl'
OUTPUT_EVAL = 'data/distill_eval.jsonl'


def convert_to_distill(input_path, output_path):
    with open(input_path) as fin, open(output_path, 'w') as fout:
        for line in fin:
            data = json.loads(line)

            # Use chosen as the teacher response
            new_data = {
                "prompt": data["prompt"],
                "response": data["chosen"]
            }

            fout.write(json.dumps(new_data) + "\n")

    print(f"Saved {output_path}")


convert_to_distill(INPUT_TRAIN, OUTPUT_TRAIN)
convert_to_distill(INPUT_EVAL, OUTPUT_EVAL)

Saved data/distill_train.jsonl
Saved data/distill_eval.jsonl


In [ ]:
# Step 4: Load model and tokenizer
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

MODEL_NAME = 'Qwen/Qwen2.5-7B-Instruct'
# Other options:
# MODEL_NAME = 'meta-llama/Llama-3.1-8B-Instruct'  # requires HF access
# MODEL_NAME = 'mistralai/Mistral-7B-Instruct-v0.3'

print(f'Loading {MODEL_NAME}...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map='auto',
    quantization_config=bnb_config,
)
print('Model loaded.')
print(f'GPU memory used: {torch.cuda.memory_allocated() / 1e9:.1f} GB')

ModuleNotFoundError: No module named 'transformers'

In [ ]:
from datasets import Dataset

def load_distill_data(path):
    records = []
    with open(path) as f:
        for line in f:
            data = json.loads(line)
            records.append({
                "prompt": data["prompt"],
                "response": data["response"],
            })
    return Dataset.from_list(records)


train_dataset = load_distill_data('data/distill_train.jsonl')
eval_dataset = load_distill_data('data/distill_eval.jsonl')

def format_example(example):
    return {
        "text": example["prompt"] + "\n\n" + example["response"]
    }

train_dataset = train_dataset.map(format_example)
eval_dataset = eval_dataset.map(format_example)

print(f'Train: {len(train_dataset)} | Eval: {len(eval_dataset)}')

In [ ]:
from peft import LoraConfig
from transformers import TrainingArguments
from trl import SFTTrainer

# LoRA config
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=['q_proj', 'v_proj', 'k_proj', 'o_proj'],
    task_type='CAUSAL_LM',
)

training_args = TrainingArguments(
    output_dir='checkpoints/role-conditioned-distill',

    # higher LR than DPO
    learning_rate=5e-6,

    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,

    logging_steps=10,
    eval_strategy='steps',
    eval_steps=50,
    save_steps=100,

    bf16=True,
    report_to='none',
    gradient_checkpointing=True,
)

print('Config ready.')

print(f'  Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}')

In [ ]:
train_dataset = train_dataset.remove_columns(
    [c for c in train_dataset.column_names if c != "text"]
)

eval_dataset = eval_dataset.remove_columns(
    [c for c in eval_dataset.column_names if c != "text"]
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    args=training_args,
    processing_class=tokenizer,
    peft_config=peft_config,
    formatting_func=lambda x: x["text"],
    
)

print('Starting self distillation training...')
trainer.train()
print('Training complete!')

In [ ]:
# Step 8: Save model
OUTPUT_DIR = 'checkpoints/role-conditioned-distill-final'
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f'Model saved to {OUTPUT_DIR}')

# Download the LoRA adapter weights
!zip -r role_conditioned_dpo_lora.zip {OUTPUT_DIR}
files.download('role_conditioned_self_distillation_lora.zip')

In [ ]:
# Step 9: Quick inference test
from peft import PeftModel

test_prompt = """You are a middle manager in engineering, who outranks the person they are speaking with. You have 1-5 years at the company.

Interaction context:
You are assigning a task to someone. Describe the task clearly, set expectations, and provide any necessary context.

Situation: The team needs to migrate the authentication service from the legacy monolith to a microservice architecture before the Q3 deadline.
Background: The migration has been planned for months but keeps getting deprioritized. The engineer you're assigning this to is capable but has a full plate.

Respond appropriately given your role and the situation."""

inputs = tokenizer(test_prompt, return_tensors='pt').to(model.device)
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        temperature=0.7,
        do_sample=True,
    )
response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
print('Generated response:')
print(response)